In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge
## Deep Learning & Generative AI Course Project

### Student Details
## **Name: Varun Choudhary**

**Roll Number:** 24F2000187
---

## Project Overview

This project focuses on solving complex Multiple Choice Questions (MCQs) using Deep Learning and Transformer-based Natural Language Processing techniques.

The objective is to predict the top three most probable answer options among five candidate answers (A, B, C, D, and E). The project explores multiple approaches ranging from traditional NLP baselines to modern Transformer architectures, along with Retrieval-Augmented Generation (RAG) concepts and parameter-efficient fine-tuning.

The final model is evaluated using Mean Average Precision at 3 (MAP@3), the official metric of the Kaggle competition.

## Problem Statement

The Smart MCQ Solver Challenge aims to build an intelligent system capable of solving challenging multiple-choice questions.

Each sample contains

- A question prompt
- Five answer options (A–E)

The task is not only to identify the correct answer but to rank the three most likely answers in descending order of confidence.

The competition is evaluated using Mean Average Precision at 3 (MAP@3), rewarding models that rank the correct answer as high as possible.

---

## Objectives

- Explore the dataset through exploratory data analysis.
- Build multiple NLP models for answer prediction.
- Fine-tune Transformer-based language models.
- Compare multiple architectures using Weights & Biases.
- Improve ranking performance through experimentation and ensembling.

##  Libraries

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForMultipleChoice,
    TrainingArguments, Trainer
)
from dataclasses import dataclass
from typing import Optional, Union
from sklearn.model_selection import train_test_split

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "microsoft/deberta-v3-large"
OPTIONS = ['A', 'B', 'C', 'D', 'E']
MAX_LEN = 256

In [ ]:
# # Login to W&B
# wandb.login()

# # Initialize project
# wandb.init(project="23F200XXXX-t22026", 
#            name="Project_Setup", 
#            config={"competition": "Smart MCQ Solver", 
#                    "framework": "PyTorch", 
#                    "evaluation_metric": "MAP@3"
#                   }
#           )

##  Load Dataset

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("Train.shape", train.shape)
print("Test.shape", test.shape)
print("Sample_submission.shape", sample_submission.shape)

Train.shape (2000, 8)
Test.shape (500, 7)
Sample_submission.shape (500, 2)


## Dataset Overview

In [3]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
train.tail()

,id,prompt,A,B,C,D,E,answer
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B
1999,2000,Pick the best possible answer: Which of the fo...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,C


In [9]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [5]:
print("\nTrain columns:", train.columns.tolist())


Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [7]:
train.nunique()

id        2000
prompt    1758
A          316
B          328
C          303
D          318
E          320
answer       5
dtype: int64

## Label Encoding & Split

In [10]:
train['label'] = train['answer'].apply(lambda x: OPTIONS.index(x))

train_df, val_df = train_test_split(
    train, test_size=0.15, random_state=SEED, stratify=train['label']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", train_df.shape, "Val:", val_df.shape)

Train: (1700, 9) Val: (300, 9)


## Tokenizer

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [12]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, has_label=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        first_sentences = [prompt] * 5
        second_sentences = [str(row[opt]) for opt in OPTIONS]

        tokenized = self.tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            max_length=self.max_len,
            padding=False,
        )
        item = {k: v for k, v in tokenized.items()}
        if self.has_label:
            item['label'] = int(row['label'])
        return item

In [13]:
train_ds = MCQDataset(train_df, tokenizer, has_label=True)
val_ds = MCQDataset(val_df, tokenizer, has_label=True)
test_ds = MCQDataset(test, tokenizer, has_label=False)

In [14]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: AutoTokenizer
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else None
        labels = [f.pop(label_name) for f in features] if label_name else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = [
            [{k: v[i] for k, v in f.items()} for i in range(num_choices)]
            for f in features
        ]
        flattened = sum(flattened, [])

        batch = self.tokenizer.pad(
            flattened,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

In [15]:
def map_at_3(preds, labels):
    top3 = np.argsort(-preds, axis=1)[:, :3]
    score = 0.0
    for p, l in zip(top3, labels):
        if l in p:
            score += 1.0 / (list(p).index(l) + 1)
    return score / len(labels)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    map3 = map_at_3(preds, labels)
    acc = (np.argmax(preds, axis=1) == labels).mean()
    return {"map@3": map3, "accuracy": acc}

In [16]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                 

In [17]:
args = TrainingArguments(
    output_dir="./mcq_deberta_large",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,
    report_to="none",
    logging_steps=20,
    save_total_limit=1,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
val_preds = trainer.predict(val_ds)
val_map3 = map_at_3(val_preds.predictions, val_df['label'].values)
print(f"Validation MAP@3: {val_map3:.4f}")

In [ ]:
test_preds = trainer.predict(test_ds)
test_logits = test_preds.predictions

top3_idx = np.argsort(-test_logits, axis=1)[:, :3]
predictions = [' '.join([OPTIONS[i] for i in row]) for row in top3_idx]

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})
submission.to_csv("submission.csv", index=False)
submission.head()